# Cloud Workload Forecasting Challenge
## Exploratory Data Analysis & Workload Forecasting Pipeline

This notebook presents an end-to-end analysis of the Microsoft Azure Public Dataset V2 (`data-v1` release).
We explore workload distributions, temporal autocorrelation, VM segmentation, and evaluate candidate forecasting models under strict leakage-safe expanding-window cross-validation.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from cwfc.data import (
    load_cpu_readings,
    load_vm_metadata,
    build_aggregate_targets,
    split_train_holdout,
)
from cwfc.analysis import (
    compute_workload_summary,
    analyze_workload_segments,
    analyze_autocorrelation,
)
from cwfc.forecasting import (
    persistence_forecast,
    moving_average_forecast,
    mean_reverting_forecast,
)
from cwfc.evaluation import (
    evaluate_temporal_cv,
    evaluate_peak_performance,
)
from cwfc.metrics import mae, rmse, smape

print('Imports successful.')


## 1. Data Ingestion & Target Aggregation
We load the VM-level readings and metadata, and construct the two required aggregate targets:
1. `active_vm_count`: count of unique VM IDs observed at each timestamp.
2. `aggregate_cpu_utilization`: mean of VM-level `avg_cpu` values at each timestamp.

In [2]:
readings = load_cpu_readings('challenge_data/vm_cpu_readings.csv.gz')
metadata = load_vm_metadata('challenge_data/vm_metadata.csv.gz')
targets = build_aggregate_targets(readings)

print(f'Readings rows: {len(readings):,}')
print(f'Unique VMs: {readings["vm_id"].nunique():,}')
print(f'Aggregate Timestamps: {len(targets)}')
targets.head()


## 2. Workload Distribution & VM Segmentation
We examine the distribution of CPU utilization across individual VMs and compare profiles between Interactive and Delay-insensitive workload categories.

In [3]:
summary = compute_workload_summary(readings, metadata)
print(f'Median VM CPU: {summary["avg_cpu_median"]:.2f}%')
print(f'Fraction < 5% CPU: {summary["idle_fraction_lt_5pct"]*100:.2f}%')
print(f'Fraction > 50% CPU: {summary["high_util_fraction_gt_50pct"]*100:.2f}%')

seg_df = analyze_workload_segments(readings, metadata)
seg_df


## 3. Autocorrelation & Time Series Dynamics
We inspect the serial correlation structure of the training series (first 110 observations) to identify memory and periodic cycles.

In [4]:
train_df, holdout_df = split_train_holdout(targets, train_size=110, holdout_size=24)
acf_df = analyze_autocorrelation(train_df['aggregate_cpu_utilization'], max_lag=18)
acf_df.head(10)


## 4. Expanding-Window Cross-Validation
To select our production model without data leakage, we evaluate candidate models across 4 temporal validation folds strictly on the 110 training observations.

In [5]:
val_results = evaluate_temporal_cv(
    train_df, target_col='aggregate_cpu_utilization', horizon=24, min_train_size=50, step=12
)
for name, stats in val_results.items():
    print(f"{name:20s} | MAE: {stats['mae_mean']:.4f} | RMSE: {stats['rmse_mean']:.4f} | Rel MAE: {stats['rel_mae']:.4f}")


## 5. Retrospective Holdout Evaluation
Finally, we retrospectively evaluate our frozen predictions against the true holdout targets.

In [6]:
preds = pd.read_csv('predictions.csv')
h_true = holdout_df['aggregate_cpu_utilization'].values
h_pred = preds['aggregate_cpu_utilization'].values

print(f"Holdout MAE:  {mae(h_true, h_pred):.4f}")
print(f"Holdout RMSE: {rmse(h_true, h_pred):.4f}")
print(f"Holdout sMAPE: {smape(h_true, h_pred):.3f}%")
